# ML-04 — Search Intelligence Data Contract

This notebook turns a March 2026 warehouse partition into a small, explicit contract for a search/content decision. The target is deliberately narrow: identify content/client pairs whose GA4 pageviews decline in a later window, using only information available before that later window.

The warehouse partition is read locally when present (the raw gated data is not committed). If the local partition is absent, the same code can read the gated Hugging Face path when `HF_TOKEN` is available in the runtime secret store. No token is stored in this notebook.

## 1. Unit of analysis + time window

**Contract in five plain-language answers:**

1. **One row:** one pseudonymized content item for one pseudonymized client, summarized from daily facts.
2. **Tables:** the March 2026 `fact_content_daily_performance` partition; no dimension table is needed for this narrow contract.
3. **Time window:** features cover March 1–10, 2026; the decision moment is after March 10; the outcome window is March 22–31, with March 11–21 left out.
4. **What we predict:** whether total GA4 pageviews decline in the outcome window versus the feature window (`label_future_pageviews_declined`), a directional decision-support proxy.
5. **Deliberate exclusion:** any future-window field, especially future pageviews and the derived label, is excluded because it is not knowable at the decision moment.

The decision moment is after March 10. A pair is labeled `1` when its total GA4 pageviews in March 22–31 are lower than its total GA4 pageviews in March 1–10. This is a directional decision-support label, not a causal claim.


In [1]:
from pathlib import Path
import os
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

MONTH = "2026-03"
LOCAL_PATH = Path("data/raw/fact_content_daily_performance_2026-03.parquet")
con = duckdb.connect()

if LOCAL_PATH.exists():
    REL = f"read_parquet('{LOCAL_PATH.as_posix()}')"
    data_source = "local March partition downloaded from the gated warehouse"
else:
    # In Colab or another clean runtime, set HF_TOKEN in the secret manager/env.
    if not os.environ.get("HF_TOKEN"):
        raise FileNotFoundError("The local partition is absent. Provide HF_TOKEN through the runtime secret store; never paste it into a cell.")
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [os.environ["HF_TOKEN"]])
    REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
    data_source = "gated Hugging Face March partition"

print(f"Source: {data_source}; month={MONTH}")


Source: local March partition downloaded from the gated warehouse; month=2026-03


## 2. Fields: feature / label / context / excluded

**Features (all knowable before the decision moment):**

- `past10_gsc_impressions`: Search Console impressions summed over March 1–10; available after that window closes.
- `past10_gsc_clicks`: Search Console clicks summed over March 1–10; available after that window closes.
- `past10_avg_position`: impression-weighted Search Console position over March 1–10; available after that window closes.
- `past10_ga4_pageviews`: GA4 pageviews summed over March 1–10; available after that window closes.
- `past10_sessions_organic`: organic sessions summed over March 1–10; available after that window closes.

**Label / proxy:** `label_future_pageviews_declined`, computed from March 22–31 pageviews versus March 1–10 pageviews. It is the outcome to predict, never a feature.

**Context:** `client_hash_id`, `content_hash_id`, and `report_date` are used for grouping, joining, and windowing only. They are not model inputs.

**Excluded:** `future_pv`, `label_future_pageviews_declined`, and any March 22–31 field are excluded from features because they are only known after the decision moment. Availability flags are used as gates, not predictive features. A label-derived column is added once below only to demonstrate leakage, then removed.

In [2]:
# No modeling happens in this cell; it records the contract's five-field boundary.
FEATURES = [
    "past10_gsc_impressions",
    "past10_gsc_clicks",
    "past10_avg_position",
    "past10_ga4_pageviews",
    "past10_sessions_organic",
]
print(f"Maximum feature count: {len(FEATURES)}")
print("Feature availability: all five summarize March 1–10 and are used only after that window closes.")


Maximum feature count: 5
Feature availability: all five summarize March 1–10 and are used only after that window closes.


## 3. Verify it with exactly three verification queries

The next three cells are the only verification queries. They check (1) grain, (2) the March slice's count and date span, and (3) availability using an explicit `IS TRUE` filter.

### Verification query 1 — grain

A valid daily fact grain should have no duplicate `(report_date, client_hash_id, content_hash_id)` keys.

In [3]:
verification_query_1 = f"""
SELECT COUNT(*) AS duplicate_grain_groups
FROM (
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {REL}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
)
"""
q1 = con.sql(verification_query_1).df()
print(q1.to_string(index=False))
assert int(q1.loc[0, "duplicate_grain_groups"]) == 0


 duplicate_grain_groups
                      0


### Verification query 2 — slice count and date span

This fixes the panel being used for all downstream work: the March 2026 partition.

In [4]:
verification_query_2 = f"""
SELECT COUNT(*) AS march_rows, MIN(report_date) AS first_date, MAX(report_date) AS last_date
FROM {REL}
"""
q2 = con.sql(verification_query_2).df()
print(q2.to_string(index=False))
assert int(q2.loc[0, "march_rows"]) > 0
assert str(q2.loc[0, "first_date"])[:10] == "2026-03-01"
assert str(q2.loc[0, "last_date"])[:10] == "2026-03-31"


 march_rows first_date  last_date
    9841378 2026-03-01 2026-03-31


### Verification query 3 — availability

The modeling slice requires GA4 availability, so rows survive only when `ga4_data_available IS TRUE`. This is a gate, not a claim that missing rows are zeros.

In [5]:
verification_query_3 = f"""
SELECT
    COUNT(*) AS total_march_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_after_ga4_is_true
FROM {REL}
"""
q3 = con.sql(verification_query_3).df()
print(q3.to_string(index=False))
assert int(q3.loc[0, "rows_after_ga4_is_true"]) > 0


 total_march_rows  rows_after_ga4_is_true
          9841378                  413966


## 4. Five-feature frame and the leakage trap

The five features are each available after the March 1–10 reporting window closes and before the future label window begins:

- `past10_gsc_impressions`: knowable at the decision moment because it is the Search Console impression total already observed on March 1–10.
- `past10_gsc_clicks`: knowable at the decision moment because it is the Search Console click total already observed on March 1–10.
- `past10_avg_position`: knowable at the decision moment because it is calculated from Search Console observations already observed on March 1–10.
- `past10_ga4_pageviews`: knowable at the decision moment because it is the GA4 pageview total already observed on March 1–10.
- `past10_sessions_organic`: knowable at the decision moment because it is the organic-session total already observed on March 1–10.

The frame below keeps only client/content pairs with GA4 available on every day in the partition and at least five observed days in each used window. It aggregates in SQL before bringing the small frame into pandas.

In [6]:
feature_sql = f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS past10_gsc_impressions,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10' THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS past10_gsc_clicks,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10' THEN COALESCE(gsc_sum_position, 0) ELSE 0 END)::DOUBLE
            / NULLIF(SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10' THEN COALESCE(gsc_impressions, 0) ELSE 0 END), 0) AS past10_avg_position,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10' THEN COALESCE(ga4_pageviews, 0) ELSE 0 END) AS past10_ga4_pageviews,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10' THEN COALESCE(sessions_organic, 0) ELSE 0 END) AS past10_sessions_organic,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10' THEN COALESCE(ga4_pageviews, 0) ELSE 0 END) AS past_pv,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31' THEN COALESCE(ga4_pageviews, 0) ELSE 0 END) AS future_pv,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10') AS past_days,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31') AS future_days,
        BOOL_AND(ga4_data_available IS TRUE) AS ga4_available_all_days
    FROM {REL}
    GROUP BY 1, 2
)
SELECT *, CAST(future_pv < past_pv AS INTEGER) AS label_future_pageviews_declined
FROM daily
WHERE ga4_available_all_days IS TRUE AND past_days >= 5 AND future_days >= 5
"""
frame = con.sql(feature_sql).df()
frame[FEATURES] = frame[FEATURES].fillna(0)
print(f"Modeling rows: {len(frame):,}; positive labels: {int(frame['label_future_pageviews_declined'].sum()):,}; decline rate: {frame['label_future_pageviews_declined'].mean():.1%}")
print(frame[FEATURES + ["label_future_pageviews_declined"]].head(5).to_string(index=False))
assert len(FEATURES) <= 5
assert len(frame) > 0


Modeling rows: 183; positive labels: 87; decline rate: 47.5%
 past10_gsc_impressions  past10_gsc_clicks  past10_avg_position  past10_ga4_pageviews  past10_sessions_organic  label_future_pageviews_declined
                 5615.0               36.0            10.626536                  71.0                     58.0                                0
                 4342.0               26.0            19.557807                  49.0                     44.0                                0
                28147.0               59.0            28.234021                 116.0                    108.0                                0
                 6768.0               75.0             5.392287                 196.0                    158.0                                0
                 8811.0               44.0            15.965725                 116.0                    105.0                                1


In [7]:
X = frame[FEATURES]
y = frame["label_future_pageviews_declined"]

def score(data, features):
    X_train, X_test, y_train, y_test = train_test_split(
        data[features], y, test_size=0.25, random_state=42, stratify=y
    )
    model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=25, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return accuracy_score(y_test, pred), roc_auc_score(y_test, proba)

# Deliberate leak: this column is literally the answer, so a near-perfect score is expected.
leaky = X.copy()
leaky["deliberate_label_derived_leak"] = y.values
leak_acc, leak_auc = score(leaky, FEATURES + ["deliberate_label_derived_leak"])
honest_acc, honest_auc = score(X, FEATURES)
print(f"With deliberate label-derived leak — accuracy={leak_acc:.3f}, ROC AUC={leak_auc:.3f}")
print(f"After removing the leak — honest accuracy={honest_acc:.3f}, ROC AUC={honest_auc:.3f}")
assert leak_acc >= 0.99 and leak_auc >= 0.99
assert "deliberate_label_derived_leak" not in FEATURES


With deliberate label-derived leak — accuracy=1.000, ROC AUC=1.000
After removing the leak — honest accuracy=0.522, ROC AUC=0.498


## Named limitation

**Panel-completeness / survivorship limitation:** the honest frame contains only content/client pairs for which GA4 is available across the March panel and enough days are observed in both windows. That makes the measured 183-pair slice useful for demonstrating the contract, but it is not representative of clients with no GA4 history or incomplete panels. The resulting score is directional decision support, not a general performance estimate for the whole warehouse.

The final model keeps five pre-decision features only. The label-derived column was used once as a controlled leakage demonstration and then removed.

## Self-check

- [x] Five plain-language contract answers are stated.
- [x] Exactly three verification queries are shown and executed.
- [x] Availability is checked with `IS TRUE`.
- [x] The feature frame has five features with an availability explanation.
- [x] The deliberate leakage experiment is shown, then removed.
- [x] One named limitation is documented.
- [x] No client names, URLs, private queries, tokens, or raw gated data are committed.